# SpectralGuard Phase 3 — Experiments
Run manually, top to bottom. Do not skip a gate. Every expensive cell resumes atomically.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('pip') is None:
    subprocess.run([sys.executable, '-m', 'ensurepip', '--upgrade'], check=True)
%pip install -q -r requirements-lightning.txt
%pip install -q -e . --no-deps

In [ ]:
import importlib.metadata as md
import torch
assert md.version('torch') == '2.7.1', f'Expected torch 2.7.1, found {md.version("torch")}'
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu_active': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})

In [ ]:
from src.experiments.phase3_runner import prepare_phase3
dry_report = prepare_phase3(mode='dry_run', materialize_assets=False)
dry_report

## Independent audit gate
Stop here. An independent reviewer must audit the exact commit and return `INDEPENDENT-AUDIT-PASS`. Only then activate the H100 and continue.

In [ ]:
audit_code = input('Independent audit code: ').strip()
assert audit_code == 'INDEPENDENT-AUDIT-PASS', 'STOP: independent audit has not passed.'
FULL_CONFIRMATION = 'I_ACCEPT_PHASE3_H100_COST'

## Activate the H100 now
The next cell verifies H100, VRAM, clean Git state, disk, versions, datasets and official snapshots.

In [ ]:
full_report = prepare_phase3(mode='full', confirmation=FULL_CONFIRMATION, materialize_assets=True)
full_report['preflight']

In [ ]:
from src.experiments.phase3_runner import run_smoke_cell, run_precision_gate
run_smoke_cell(confirmation=FULL_CONFIRMATION)
run_precision_gate(confirmation=FULL_CONFIRMATION)

In [ ]:
from src.experiments.phase3_runner import run_official_hispa_selection_cell
run_official_hispa_selection_cell(confirmation=FULL_CONFIRMATION)['selected']

In [ ]:
from src.experiments.phase3_runner import run_attack_optimization_cell
run_attack_optimization_cell('gradient_guided', confirmation=FULL_CONFIRMATION)['best']['final_diagnostics']

In [ ]:
run_attack_optimization_cell('adaptive_v2', confirmation=FULL_CONFIRMATION)['best']['final_diagnostics']

In [ ]:
from src.experiments.phase3_runner import run_model_cell
run_model_cell('E04_adaptive', confirmation=FULL_CONFIRMATION)

In [ ]:
from src.experiments.phase3_runner import fit_detector_cell
fit_detector_cell()['status']

In [ ]:
run_attack_optimization_cell('adaptive_v3', confirmation=FULL_CONFIRMATION)['best']['final_diagnostics']
run_attack_optimization_cell('pareto', confirmation=FULL_CONFIRMATION)['restart_count']

In [ ]:
# Resume E04 and add adaptive_v3 without repeating completed pairs.
run_model_cell('E04_adaptive', confirmation=FULL_CONFIRMATION)

In [ ]:
for experiment_id in ['E01_main_mamba', 'E02_gpt2_control', 'E03_pythia_control', 'E05_transfer_mamba_1_4b', 'E06_transfer_mamba_2_8b', 'E07_transfer_pythia_1_4b', 'E08_zamba']:
    print(experiment_id, run_model_cell(experiment_id, confirmation=FULL_CONFIRMATION))

In [ ]:
from src.experiments.phase3_runner import export_experiment_bundle
experiments_zip = export_experiment_bundle()
print(experiments_zip)